### Problem: Calculate the Required Sample Size for an A/B Test
You are preparing an A/B test to see if a new checkout button increases the conversion rate. Before running the test, you must calculate the required sample size per variant. We will use a historical online dataset to calculate the Baseline Conversion Rate (BCR), then compute the required sample size for an 80% statistical power, a 5% significance level, and an expected absolute Minimum Detectable Effect (MDE) of 1% (0.01).

**Sample Input (Historical Data)**:
| user_id | timestamp | group | landing_page | converted |
|---|---|---|---|---|
| 851104 | 2017-01-21 22:11:48.556739 | control | old_page | 0 |
| 804228 | 2017-01-12 08:01:45.159739 | control | old_page | 0 |
| 661590 | 2017-01-11 16:55:06.154213 | treatment | new_page | 0 |

**Sample Output**:
```
Baseline Conversion Rate: 11.97%
Required Sample Size per Variant: 15,861
```

**What you should use:**
- `pandas` to read the CSV from the public URL and calculate the baseline conversion rate (mean of the `converted` column).
- `statsmodels.stats.api.proportion_effectsize` to calculate the standardized effect size between the baseline and the expected new conversion rate.
- `statsmodels.stats.power.NormalIndPower().solve_power` to calculate the actual sample size based on the effect size, alpha (0.05), power (0.80), and ratio (1.0).

In [6]:
import pandas as pd
import statsmodels.stats.api as sms
import scipy.stats as stats

# 1. Fetch the dataset
url = 'https://raw.githubusercontent.com/ozlerhakan/ab-test/master/ab_data.csv'
df = pd.read_csv(url)

# View the first few rows to understand the data
df.shape
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


---
### Optimized Solution

In [ ]:
import pandas as pd
import statsmodels.stats.api as sms
import math

# 1. Fetch data and establish Baseline Conversion Rate (BCR)
url = 'https://raw.githubusercontent.com/ozlerhakan/ab-test/master/ab_data.csv'
df = pd.read_csv(url)

# The historical conversion rate is our baseline
bcr = df['converted'].mean()

# 2. Define the Test Parameters
mde = 0.01         # Minimum Detectable Effect (Absolute 1% increase)
alpha = 0.05       # Significance Level (5%)
power = 0.80       # Statistical Power (80%)

# 3. Calculate the Effect Size
expected_cr = bcr + mde
effect_size = sms.proportion_effectsize(bcr, expected_cr)

# 4. Calculate Sample Size
sample_size = sms.NormalIndPower().solve_power(
    effect_size=effect_size, 
    power=power, 
    alpha=alpha, 
    ratio=1.0
)

# Round up as we need whole users
required_n = math.ceil(sample_size)

print(f"Baseline Conversion Rate: {bcr:.4%}")
print(f"Expected Conversion Rate: {expected_cr:.4%}")
print(f"Required Sample Size per Variant: {required_n:,}")

---
### Concepts Explained (Step-by-Step)

**1. Baseline Conversion Rate (BCR)**
*   **What it is:** The current success rate of your website *before* you make any changes.
*   **Why we need it:** If your website currently converts at 80%, it will take very few users to detect a 1% jump compared to a website that currently converts at 2%. Statistics requires knowing your starting point.

**2. MDE (Minimum Detectable Effect)**
*   **What it is:** The smallest difference that actually matters to the business. 
*   **Example:** If your conversion rate is 11%, an absolute MDE of 1% means you are trying to detect if the new rate hits 12%. If the new button only increases conversion by a microscopic 0.001%, the business probably doesn't care—it's not worth the engineering effort to deploy it. 

**3. Alpha ($\alpha$) - Statistical Significance**
*   **What it is:** The probability of a **False Positive** (Type I Error). 
*   **Plain English:** It is the risk you take of saying "the new button is better!" when, in reality, it's not (it was just random luck). By setting it to 5% (0.05), you are saying you demand a 95% confidence level before you declare a winner.


**3b. P-Value (And its relationship to Alpha)**
*   **What it is:** The probability of seeing a result as extreme as (or more extreme than) what you observed, *assuming the null hypothesis is true* (i.e., assuming there is no real difference).
*   **The Relationship:** Think of Alpha ($\alpha$) as the **threshold** you set *before* the test (e.g., 0.05). Think of the P-value as the **result** you get *after* the test. 
*   **The Decision Rule:** If **P-value < Alpha**, you reject the null hypothesis and declare the result statistically significant. You are essentially saying: *"The probability that this happened by random chance (p-value) is lower than the maximum risk I am willing to tolerate (Alpha). Therefore, the effect must be real."*

**4. Power ($1 - \beta$)**
*   **What it is:** The probability of correctly detecting a true difference (preventing a **False Negative** or Type II Error).
*   **Plain English:** If the new button *actually is* 1% better, an 80% power means there is an 80% chance your test will successfully catch it, and a 20% chance your test will completely miss it and tell you it failed. 80% is the industry standard.

**5. Effect Size**
*   **What it is:** Effect size is a standardized way to measure "how big a difference" you are looking for, completely independent of the actual numbers. The statistical formula doesn't care if you are measuring conversion rates, height in inches, or money. It just needs a standardized, unitless number that represents the distance between two groups. 
*   **The Golden Rule:** The *smaller* the effect size you want to detect (e.g., trying to detect a 0.1% change instead of a 5% change), the *larger* the sample size (more users) you will need to reliably prove it wasn't just random chance.

**6. The Final Calculation (`sms.NormalIndPower().solve_power`)**
*   **Plain English:** You are asking the calculator: *"I want to be 95% confident (`alpha`) and have an 80% chance of catching the change (`power`). The gap I want to detect is standardized as `effect_size`. My Control and Treatment groups will split traffic 50/50 (`ratio=1.0`). How many users do I need in each group?"* The function then spits out the exact number—for instance, 15,861 users per group.
